In [1]:
# Install package
!pip install -q tensorflow pyngrok flask flask-cors

# Import
from flask import Flask, request, jsonify
from flask_cors import CORS
from werkzeug.utils import secure_filename

import tensorflow as tf
from tensorflow.keras.preprocessing.image import (
    load_img,
    img_to_array
)

from pyngrok import ngrok
from google.colab import drive

import numpy as np
import os

In [2]:
#Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Authenticate ngrok
NGROK_AUTH_TOKEN = '3FNuhvCGfsK0pQKngALVIMfSekZ_3RVDRHHHZAqDXAhAvtyrX'
# Replace with your ngrok auth token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

from flask_cors import CORS
# Create Flask app
app = Flask(__name__)
CORS(app)

# Load the model
model_path = '/content/drive/MyDrive/Colab Notebooks/KlasifikasiPatahTulang/bone_cracking_classification_model.keras'
model = tf.keras.models.load_model(
    model_path,
    compile=False
)

# Define root route
@app.route('/', methods=['GET'])
def index():
    return "<h1>Hello World</h1>"

# Define prediction route
@app.route('/predict', methods=['POST'])
def upload():
    data = {"success": False}

    if 'file' not in request.files:
        data["file"] = "Tidak ada file"
        return jsonify(data)

    file = request.files['file']

    if file.filename == '':
        data["file"] = "Tidak ada file"
        return jsonify(data)

    # Secure the filename and save it
    filename = secure_filename(file.filename)
    filepath = os.path.join('/content/drive/MyDrive/DATA MINING/klasifikasi-patah-tulang/test', filename)
    file.save(filepath)

    try:
        # Load and preprocess the image
        img = load_img(filepath, target_size=(224, 224))
        x = img_to_array(img)
        x = x.reshape((1,) + x.shape)
        x /= 255.0

        # Make a prediction
        predict = model.predict(x)

        # Interpret the prediction
        temp = 0
        label = 0
        hasil = []

        for y in range(7):
            presentase = predict[0][y] * 100
            hasil.append(presentase)
            if presentase > temp:
                temp = presentase
                label = y

        data["success"] = True
        data["label"] = int(label)
        data["confidence"] = float(temp)
        data["predictions"] = [float(p) for p in hasil]

    except Exception as e:
        data["error"] = str(e)

    return jsonify(data)

if __name__ == '__main__':
    # Ensure the 'data_tes' directory exists
    if not os.path.exists('/content/drive/MyDrive/DATA MINING/klasifikasi-patah-tulang/test'):
        os.makedirs('/content/drive/MyDrive/DATA MINING/klasifikasi-patah-tulang/test')

    # Start ngrok tunnel
    public_url = ngrok.connect(5000)
    print(f'Public URL: {public_url}')

    # Run the Flask app
    app.run()

Public URL: NgrokTunnel: "https://nape-emphasis-improvise.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 900ms/step


INFO:werkzeug:127.0.0.1 - - [02/Jul/2026 01:51:59] "POST /predict HTTP/1.1" 200 -
